# Demo | Capítulo 14: Regresión Lineal Simple
**Equipo:** Loompy — Grus, Cap. 14

Modelamos el **precio de departamentos en venta en Capital Federal** (`l2 = "Capital Federal"`) con una recta. Comparamos varias columnas como variable $x$ y nos quedamos con la que mejor ajusta según $R^2$.

## Índice
1. Setup y carga del dataset
2. Preparación de datos
3. El modelo de regresión lineal (teoría + funciones)
4. Error de predicción y RSS
5. Mínimos cuadrados
6. Coeficiente de determinación $R^2$
7. Comparación por $R^2$: selección de variable e interpretación
8. Mejor variable: ajuste, predicciones y $R^2$
9. Descenso del gradiente
10. Estimación de máxima verosimilitud
11. Visualización
12. Gotchas / anti-patrones
13. Cierre

# El modelo de regresión lineal

### Introducción
Presentamos la teoría de la Regresión Lineal Simple, adaptando los conceptos estadísticos al análisis del mercado inmobiliario argentino con el conjunto de datos de Properati.

Nuestro objetivo es comprender y modelar la relación lineal existente entre:

**Variable independiente ($x$):** un atributo de la propiedad (por ejemplo superficie en $m^2$, ambientes, baños). Más adelante comparamos varias candidatas y nos quedamos con la de mayor $R^2$.

**Variable dependiente ($y$):** el precio normalizado de la propiedad (en este recorte, **ARS**).

El modelo de regresión lineal analiza la relación entre una variable independiente ($x$) y una dependiente ($y$). Asume que el precio está determinado principalmente por $x$, según:

$\qquad y_i = \alpha + \beta x_i + \epsilon_i$

Donde:
*   $y_i$ (**precio real**): precio observado (ya en ARS) del anuncio $i$.
*   $x_i$ (**predictor**): valor de la variable elegida para esa propiedad (p. ej. $m^2$).
*   $\beta$ (**pendiente**): cambio esperado en el precio por cada unidad adicional de $x$. Si $x$ es superficie, es el costo incremental por $m^2$.
*   $\alpha$ (**intersección**): precio base teórico cuando $x = 0$. En la práctica absorbe costos fijos; **no** hay que leerlo como el precio real de un depto de $0\,m^2$.
*   $\epsilon_i$ (**error o residuo**): lo que el modelo no captura con $x$ sola (ubicación, antigüedad, cochera, amenities, calidad).

Con $\alpha$ y $\beta$ estimados, la predicción es:

$\qquad \hat{y}_i = \alpha + \beta x_i$

# SETUP
Antes de correr los notebooks, instalá las dependencias desde la terminal:

```bash
pip install -r requirements.txt
```

(Si estás en Kaggle Notebooks o Google Colab, no hace falta este paso.) (Si querés correrlo en el notebook, agregá `!` antes.)

# SETUP → Dataset

Usamos el dataset de Properati Argentina:

https://www.kaggle.com/datasets/alejandroczernikier/properati-argentina-dataset

La carga busca el CSV en este orden: `data/` local → Kaggle Notebooks → descarga con `kagglehub`.

In [ ]:
from pathlib import Path
import json
import os
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams["figure.figsize"] = (8, 5)

COLOR_PRINCIPAL = "#a72424"
COLOR_SECUNDARIO = "#df6060"


def cargar_properati():
    # Carga el dataset Properati desde distintas fuentes, según el entorno.

    # 1) CSV local en data/ (repo clonado; un nivel arriba desde notebooks/)
    ruta_local = Path("../data/entrenamiento.csv")
    if ruta_local.exists():
        print("Cargando desde data/ local:", ruta_local)
        return pd.read_csv(ruta_local, low_memory=False)

    # 2) Kaggle Notebooks (al adjuntar el dataset hay pasos guiados)
    ruta_kaggle = Path(
        "/kaggle/input/datasets/alejandroczernikier/properati-argentina-dataset/entrenamiento.csv"
    )
    if ruta_kaggle.exists():
        print("Cargando desde Kaggle Notebooks")
        return pd.read_csv(ruta_kaggle, low_memory=False)

    # 3) kagglehub (Kaggle, Colab y local, con credenciales configuradas)
    try:
        import kagglehub

        path = kagglehub.dataset_download("alejandroczernikier/properati-argentina-dataset")
        print(f"Descargado con kagglehub en: {path}")
        return pd.read_csv(os.path.join(path, "entrenamiento.csv"), low_memory=False)
    except Exception as e:
        raise FileNotFoundError(
            "No se encontró el dataset en ninguna fuente. "
            "Revisá el README para instrucciones de descarga según tu entorno. "
            f"Error original: {e}"
        )


datos_crudos = cargar_properati()
print("Filas totales:", len(datos_crudos))
print("Shape:", datos_crudos.shape)
datos_crudos.head()

## 2. Preparación de datos

- Solo **Capital Federal** (`l2 == "Capital Federal"`)
- Solo **departamentos en venta**
- Precio normalizado a **ARS** (API BCRA si está en USD), para que $y$ sea comparable
- Recorte de precios extremos y varias columnas numéricas para probar después

Sin este recorte, mezclaríamos alquileres con ventas, casas con departamentos y pesos con dólares: la recta dejaría de describir un mercado y pasaría a describir un revoltijo.

In [ ]:
URL_BCRA = "https://api.bcra.gob.ar/estadisticascambiarias/v1.0/Cotizaciones/USD"
CIUDAD = "Capital Federal"
PRECIO_MIN, PRECIO_MAX = 500_000, 80_000_000


def cotizaciones_mensuales(desde, hasta):
    # Consulta la API del BCRA y arma el tipo de cambio USD→ARS promedio por mes.
    url = f"{URL_BCRA}?fechadesde={desde}&fechahasta={hasta}&limit=1000"
    req = urllib.request.Request(url, headers={"User-Agent": "loompy-lab1"})
    with urllib.request.urlopen(req, timeout=30) as resp:
        payload = json.loads(resp.read().decode())
    filas = [
        {"fecha": pd.to_datetime(i["fecha"]), "cotizacion": i["detalle"][0]["tipoCotizacion"]}
        for i in payload["results"]
    ]
    t = pd.DataFrame(filas).sort_values("fecha").set_index("fecha")["cotizacion"]
    mensual = t.resample("MS").mean()
    mensual.index = mensual.index.strftime("%Y-%m")
    return mensual


fechas = pd.to_datetime(datos_crudos["created_on"])
cotiz = cotizaciones_mensuales(fechas.min().strftime("%Y-%m-%d"), fechas.max().strftime("%Y-%m-%d"))
print("Cotizaciones USD→ARS (mensual):")
print(cotiz.round(2))
print()


def tc_mes(fecha):
    # Cotización del mes de una fecha (si falta, usa la última disponible).
    return float(cotiz.get(fecha.strftime("%Y-%m"), cotiz.iloc[-1]))


def precio_en_pesos(fila):
    # Deja ARS igual; multiplica USD por el TC del mes.
    if fila["currency"] == "ARS":
        return fila["price"]
    if fila["currency"] == "USD":
        return fila["price"] * tc_mes(fila["created_on"])
    return np.nan


datos = datos_crudos.copy()
datos["created_on"] = pd.to_datetime(datos["created_on"])

datos = datos[datos["l2"] == CIUDAD]
print(f"Filas en {CIUDAD}:", len(datos))

datos = datos[
    (datos["property_type"] == "Departamento") & (datos["operation_type"] == "Venta")
]
print("Depto + venta:", len(datos))

datos = datos[datos["price"] > 0]
datos["precio_ars"] = datos.apply(precio_en_pesos, axis=1)
datos = datos.dropna(subset=["precio_ars"])
datos = datos[(datos["precio_ars"] >= PRECIO_MIN) & (datos["precio_ars"] <= PRECIO_MAX)]

print(f"\nFilas listas ({CIUDAD}, depto + venta + precio OK):", len(datos))

## 3. Funciones del modelo

Implementación desde cero (Grus, cap. 14), sobre el recorte de Capital Federal.

In [ ]:
def predecir(intercepto, pendiente, x):
    # y = pendiente * x + intercepto
    return pendiente * x + intercepto


def suma_errores_cuadrado(intercepto, pendiente, x, y):
    # SSE / RSS: qué tan lejos están las predicciones de y.
    return np.sum((predecir(intercepto, pendiente, x) - y) ** 2)


def media(v):
    return np.mean(v)


def desviacion(v):
    return np.std(v)


def correlacion(x, y):
    # Pearson: qué tan linealmente se mueven juntas x e y.
    return np.corrcoef(x, y)[0, 1]


def ajustar_minimos_cuadrados(x, y):
    # Recta OLS: (intercepto alpha, pendiente beta).
    beta = correlacion(x, y) * desviacion(y) / desviacion(x)
    alpha = media(y) - beta * media(x)
    return alpha, beta


def r_cuadrado(intercepto, pendiente, x, y):
    # Proporción de la varianza de y explicada por la recta (1 = ajuste perfecto).
    sse = suma_errores_cuadrado(intercepto, pendiente, x, y)
    sst = np.sum((y - media(y)) ** 2)
    return 1 - sse / sst


print("Funciones listas.")

## 4. El error de predicción y la suma de errores al cuadrado (RSS)

En la realidad, las propiedades no se tasan con perfección matemática; siempre hay diferencia entre lo que el modelo predice y el precio publicado. Esa diferencia es el error (residuo) de un anuncio:

$\qquad \text{Error}_i = y_i - \hat{y}_i = y_i - (\alpha + \beta x_i)$

Para que errores positivos y negativos no se cancelen, se eleva cada diferencia al cuadrado y se suman:

$\qquad \text{RSS} = \sum_{i=1}^{n} (y_i - (\alpha + \beta x_i))^2$

*   **Penalización de desvíos grandes:** errar por 10 se convierte en 100; errar por 100, en 10.000. El modelo evita tasaciones extremadamente erróneas.
*   **Facilidad matemática:** al eliminar signos negativos, la función es suave y se puede minimizar con cálculo (mínimos cuadrados) o de forma iterativa (gradiente).

### 5. El método de mínimos cuadrados

Busca los $\alpha$ y $\beta$ que dejan el RSS lo más bajo posible. Las fórmulas cerradas, a partir de los datos, son:

**Pendiente** ($\beta$):

$\qquad \beta = \text{correlación}(x,y) \frac{\text{desviación estándar}(y)}{\text{desviación estándar}(x)}$

La correlación mide intensidad y dirección de la relación lineal: cerca de 1, al subir $x$ sube el precio de forma predecible; cerca de 0, no hay recta clara.
La razón de desvíos convierte esa relación en unidades de $y$ por unidad de $x$ (en nuestro caso, ARS por $m^2$, por ambiente, etc.).

**Intersección** ($\alpha$):

$\qquad \alpha = \text{media}(y) - \beta \cdot \text{media}(x)$

Así, una propiedad con $x$ igual al promedio del recorte recibe un precio predicho igual al precio promedio del mercado local.

### 6. El coeficiente de determinación ($R^2$)

$R^2$ compara lo que el modelo no explica con la variación total de precios:

*   **RSS:** error que queda "suelto".
*   **TSS:** suma de cuadrados de $y_i$ respecto de la media de $y$.

$\qquad R^2 = 1.0 - \frac{\text{RSS}}{\text{TSS}}$

*   **$R^2 = 0$:** la recta no aporta; conviene predecir con la media.
*   **$R^2 = 1$:** la recta explica toda la variación; no hay residuos.

**Interpretación:** un $R^2$ de 0.65 indica que el 65% de la variación de precios se explica con $x$, y el 35% restante queda en ubicación, amenities, antigüedad, etc. Eso es esperable en un modelo de **una sola** variable.

## 7. Comparación por $R^2$: selección de variable

Criterio de selección: ajustar `precio_ars ~ columna` para varias $x$ candidatas y quedarnos con la de **mayor $R^2$** (también miramos la correlación).

En esta sección no armamos aún el ajuste completo: alcanza con rankear. Elegimos $R^2$ y no solo la correlación porque $R^2$ dice qué fracción del precio queda explicada por esa recta, que es lo que nos importa para tasación.

In [ ]:
# columnas a probar: nombre técnico → (etiqueta, mínimo, máximo)
CANDIDATAS = {
    "surface_total": ("Superficie total (m²)", 20, 500),
    "surface_covered": ("Superficie cubierta (m²)", 20, 500),
    "rooms": ("Ambientes", 1, 10),
    "bedrooms": ("Dormitorios", 0, 8),
    "bathrooms": ("Baños", 1, 6),
}


def ranking_columna(base, columna, etiqueta, vmin, vmax):
    sub = base.dropna(subset=[columna, "precio_ars"]).copy()
    sub = sub[(sub[columna] >= vmin) & (sub[columna] <= vmax)]

    x = sub[columna].to_numpy(dtype=float)
    y = sub["precio_ars"].to_numpy(dtype=float)
    alpha, beta = ajustar_minimos_cuadrados(x, y)

    return {
        "columna": columna,
        "nombre": etiqueta,
        "filas": len(sub),
        "correlacion": correlacion(x, y),
        "r2": r_cuadrado(alpha, beta, x, y),
        "vmin": vmin,
        "vmax": vmax,
    }


ranking = [ranking_columna(datos, col, *params) for col, params in CANDIDATAS.items()]

comparacion = pd.DataFrame(
    [
        {
            "variable": r["nombre"],
            "columna": r["columna"],
            "filas": r["filas"],
            "correlacion": round(r["correlacion"], 4),
            "r2": round(r["r2"], 4),
        }
        for r in ranking
    ]
).sort_values("r2", ascending=False).reset_index(drop=True)

print("Ranking por R² (de mayor a menor):")
comparacion

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(comparacion["variable"], comparacion["r2"], color=COLOR_PRINCIPAL)
ax.set_xlabel("R²")
ax.set_title(f"¿Qué variable explica mejor el precio en {CIUDAD}?")
ax.set_xlim(0, 1)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

### Interpretación del ranking

En este escenario (deptos en venta en CABA, precio en ARS), la superficie —total o cubierta— suele ganar: el mercado cotiza sobre todo metros. Ambientes, dormitorios y baños correlacionan con el precio pero son discretos y van de la mano de la superficie, así que su $R^2$ simple es más bajo.

**Qué muestra esto para el capítulo:** con una sola $x$, el criterio natural de selección es $R^2$ (o, equivalentemente, $|\text{correlación}|$ en el caso lineal simple). No mezclamos métodos: o bien OLS en forma cerrada, o bien gradiente como chequeo numérico. El ranking no implica causalidad: un $R^2$ alto no dice "el precio sube *porque* hay más $m^2$", sino que esa variable es la que mejor **predice** en este recorte.

## 8. Mejor variable: ajuste, predicciones y $R^2$

Elegimos la columna con mayor $R^2$ y **recién acá** armamos el dataset de esa $x$, el ajuste completo y la interpretación de $\alpha$, $\beta$ y $R^2$.

In [ ]:
mejor_rank = max(ranking, key=lambda r: r["r2"])
columna_x = mejor_rank["columna"]
nombre_x = mejor_rank["nombre"]
vmin, vmax = mejor_rank["vmin"], mejor_rank["vmax"]

sub = datos.dropna(subset=[columna_x, "precio_ars"]).copy()
sub = sub[(sub[columna_x] >= vmin) & (sub[columna_x] <= vmax)]
x = sub[columna_x].to_numpy(dtype=float)
y = sub["precio_ars"].to_numpy(dtype=float)

alpha, beta = ajustar_minimos_cuadrados(x, y)
r2 = r_cuadrado(alpha, beta, x, y)
rss = suma_errores_cuadrado(alpha, beta, x, y)

print(f"Mejor variable: {nombre_x} ({columna_x})")
print(f"Filas usadas: {len(sub):,}")
print(f"Correlación: {correlacion(x, y):.4f}")
print(f"RSS = {rss:,.0f}")
print(f"R² = {r2:.4f}  →  explica el {r2 * 100:.1f}% de la variación del precio")
print(f"Intercepto α = {alpha:,.0f}")
print(f"Pendiente  β = {beta:,.0f}")
print(f"Modelo: precio_ars ≈ {beta:,.0f} × {columna_x} + ({alpha:,.0f})")

### 9. Optimización mediante descenso del gradiente

Con cientos de miles de anuncios, calcular medias, desvíos y correlaciones en un solo paso puede ser pesado. Una alternativa de ML es estimar $\alpha$ y $\beta$ de forma **iterativa**.

Se parte de valores iniciales y, en cada paso, se mira cómo cambia el error. Las derivadas del RSS son:

*   **Respecto del precio base** ($\alpha$):

    $\qquad \frac{\partial \text{Loss}}{\partial \alpha} = \sum_{i=1}^{n} -2(\text{error}_i)$

*   **Respecto de la pendiente** ($\beta$):

    $\qquad \frac{\partial \text{Loss}}{\partial \beta} = \sum_{i=1}^{n} -2(\text{error}_i)x_i$

Actualización con tasa de aprendizaje $\eta$:

$\qquad \alpha_{\text{nuevo}} = \alpha_{\text{viejo}} - \eta \frac{\partial \text{Loss}}{\partial \alpha}$

$\qquad \beta_{\text{nuevo}} = \beta_{\text{viejo}} - \eta \frac{\partial \text{Loss}}{\partial \beta}$

Se repite hasta que los parámetros se estabilizan cerca de las fórmulas de mínimos cuadrados.

**En la práctica:** $x$ (p. ej. $m^2$) e $y$ (millones de ARS) tienen escalas muy distintas. Sin estandarizar, el gradiente no converge bien. Por eso normalizamos, descendemos, y después **deshacemos** la escala para reportar $\alpha$ y $\beta$ en unidades originales.

In [ ]:
# Estandarizamos para que converja bien
x_mu, x_desv = media(x), desviacion(x)
y_mu, y_desv = media(y), desviacion(y)
x_n = (x - x_mu) / x_desv
y_n = (y - y_mu) / y_desv

a_g, b_g = 0.0, 0.0
tasa = 0.01
historial = []

for _ in range(2000):
    res = predecir(a_g, b_g, x_n) - y_n
    ga = 2 * np.sum(res) / len(x_n)
    gb = 2 * np.sum(res * x_n) / len(x_n)
    a_g -= tasa * ga
    b_g -= tasa * gb
    historial.append(np.mean(res ** 2))

alpha_gd = y_mu + y_desv * a_g - b_g * (y_desv / x_desv) * x_mu
beta_gd = b_g * (y_desv / x_desv)

print(f"OLS:       α={alpha:,.0f}  β={beta:,.0f}")
print(f"Gradiente: α={alpha_gd:,.0f}  β={beta_gd:,.0f}")

plt.plot(historial, color=COLOR_PRINCIPAL)
plt.title("Error durante el descenso del gradiente")
plt.xlabel("Época")
plt.ylabel("MSE")
plt.show()

### 10. Estimación de máxima verosimilitud

La estimación de máxima verosimilitud (MLE) es el sustento estadístico del modelo. No conocemos los verdaderos $\alpha$ y $\beta$ del mercado; MLE busca los valores que hacen **más probables** los precios que vemos en Properati.

Asumimos que, **dentro de un mercado homogéneo** (acá: deptos en venta en CABA), los errores se distribuyen normal en torno a cero: muchos desvíos chicos, casi ninguno gigante. Si mezcláramos Puerto Madero con Soldati, o ventas con alquileres, ese supuesto se rompe.

Bajo normalidad, maximizar la verosimilitud **equivale** a minimizar el RSS. Mínimos cuadrados deja de ser un truco algebraico y pasa a ser el estimador estadísticamente óptimo de la recta de ese mercado local.

## 11. Visualización: OLS vs gradiente descendente

Las dos rectas deberían superponerse: el gradiente es otra forma de llegar al mismo mínimo del RSS.

In [ ]:
idx = np.random.default_rng(42).choice(len(x), size=min(2000, len(x)), replace=False)
xs, ys = x[idx], y[idx] / 1e6
x_line = np.linspace(x.min(), x.max(), 100)

for a, b, titulo in [
    (alpha, beta, "Mínimos cuadrados"),
    (alpha_gd, beta_gd, "Gradiente descendente"),
]:
    plt.figure()
    plt.scatter(xs, ys, alpha=0.3, s=10, color=COLOR_SECUNDARIO)
    plt.plot(x_line, predecir(a, b, x_line) / 1e6, color=COLOR_PRINCIPAL, linewidth=2)
    plt.title(f"{CIUDAD} — {titulo}\n{nombre_x}")
    plt.xlabel(nombre_x)
    plt.ylabel("Precio (millones ARS)")
    plt.tight_layout()
    plt.show()

## 12. Gotchas / anti-patrones

- **Monedas mezcladas.** Entrenar con `price` crudo (USD y ARS juntos) hace que $\beta$ no tenga unidades. Hay que normalizar (acá, a ARS con el TC del mes).
- **Mercados heterogéneos.** Juntar CABA con el interior, venta con alquiler, o depto con casa rompe la recta y el supuesto de errores normales de MLE. El recorte no es cosmética: es parte del modelo.
- **Outliers y RSS.** Un depto de $2.000\,m^2$ o un precio de 1 USD domina la suma de cuadrados. Por eso recortamos superficie y precio; hay que **declarar** ese recorte, porque cambia $R^2$.
- **Leer $\alpha$ como precio real.** $x = 0$ está fuera del rango de los datos. $\alpha$ ancla la recta; no es el valor de un terreno vacío.
- **Gradiente sin estandarizar.** Con $x$ en $m^2$ e $y$ en millones, $\eta$ no alcanza o diverge. Estandarizar → descender → volver a la escala original.
- **$R^2$ alto ≠ causalidad.** Que la superficie gane el ranking no prueba que "el precio sube *porque* hay más metros"; solo que predice mejor en este recorte. Ambientes y baños importan, pero viajan con la superficie.
- **Filtros que sesgan la muestra.** Tirar NaNs de `surface_covered` puede dejar solo avisos "completos" (a menudo más caros). El $R^2$ es del recorte, no de todo Properati.
- **Tratar variables discretas como continuas.** Baños = 1, 2, 3… La recta es una aproximación; un $R^2$ más bajo no significa que el baño no importe.

## 13. Cierre

En Capital Federal, una recta simple ya muestra que el precio de un depto en venta se explica sobre todo por la superficie: OLS y descenso del gradiente llegan al mismo $(\alpha, \beta)$, y $R^2$ sirve tanto para medir el ajuste como para **elegir** $x$.

Lo que el modelo *no* explica —barrio, amenities, antigüedad— queda en el residuo; por eso un $R^2$ intermedio es un resultado honesto, no un fracaso.

Pasamos al ejercicio guiado en `ejercicios_equipo.ipynb`.